# Speaker Diarization Pipeline Integration

**Assignment:** Wire pyannote speaker diarization into the Foreign Whispers pipeline so multi-speaker videos produce per-speaker labeled segments.

## Pipeline Flow

```
CURRENT:  Download → Transcribe → Translate → TTS → Stitch
TARGET:   Download → Transcribe → Diarize → Translate → TTS (per-speaker) → Stitch
                                    ↑
                              YOUR WORK HERE
```

## Prerequisites

- Docker Compose stack is running: `docker compose --profile nvidia up -d`
- `FW_HF_TOKEN` is set in your `.env` or environment
- You have accepted the [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) model license on HuggingFace
- A multi-speaker test video is in `video_registry.yml` and downloaded

## Provided Code (Read First)

| File | What it does |
| ---- | ------------ |
| `foreign_whispers/diarization.py` | `diarize_audio(audio_path, hf_token)` — calls pyannote, returns `[{start_s, end_s, speaker}]` |
| `api/src/services/alignment_service.py` | `AlignmentService.diarize()` — service wrapper |
| `api/src/core/config.py` | `Settings.hf_token` — reads `FW_HF_TOKEN` env var |
| `pipeline_data/speakers/` | Per-language directories with reference WAV files for TTS voice cloning |

In [10]:
# Setup: add project root to path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/joshu/CS4613/foreign-whispers


---

## Task 1: Segment-Speaker Merge Function

**Goal:** Write a pure function that assigns a speaker label to each transcription segment based on diarization output.

**Algorithm:** For each transcription segment, find which diarization speaker has the most temporal overlap with that segment's `[start, end]` range.

**File to modify:** `foreign_whispers/diarization.py` (add function to existing file)

### 1.1 — Read the existing diarization code

In [11]:
# Read the existing diarization module
diar_path = PROJECT_ROOT / "foreign_whispers" / "diarization.py"
print(diar_path.read_text())

"""Speaker diarization using pyannote.audio.

Extracted from notebooks/foreign_whispers_pipeline.ipynb (M2-align).

Optional dependency: pyannote.audio
    pip install pyannote.audio
Requires accepting the pyannote/speaker-diarization-3.1 licence on HuggingFace
and providing an HF token.  Returns empty list with a warning if the dep is
absent or the token is missing.
"""
import logging

logger = logging.getLogger(__name__)


def diarize_audio(audio_path: str, hf_token: str | None = None) -> list[dict]:
    """Return speaker-labeled intervals for *audio_path*.

    Returns:
        List of ``{start_s: float, end_s: float, speaker: str}``.
        Empty list when pyannote.audio is absent, token is missing, or diarization fails.
    """
    if not hf_token:
        logger.warning("No HF token provided — diarization skipped.")
        return []

    try:
        from pyannote.audio import Pipeline
    except (ImportError, TypeError):
        logger.warning("pyannote.audio not installed — r

### 1.2 — Write the tests first (TDD)

Run the tests below. They will **fail** because `assign_speakers` doesn't exist yet. That's expected.

In [12]:
# These tests define the contract for assign_speakers.
# DO NOT modify the tests — make your implementation pass them.

def run_tests():
    from foreign_whispers.diarization import assign_speakers
    
    passed, failed = 0, 0
    
    # Test 1: Single speaker
    try:
        segments = [
            {"id": 0, "start": 0.0, "end": 3.0, "text": "Hello world"},
            {"id": 1, "start": 3.5, "end": 6.0, "text": "How are you"},
        ]
        diarization = [
            {"start_s": 0.0, "end_s": 7.0, "speaker": "SPEAKER_00"},
        ]
        result = assign_speakers(segments, diarization)
        assert len(result) == 2
        assert result[0]["speaker"] == "SPEAKER_00"
        assert result[1]["speaker"] == "SPEAKER_00"
        assert result[0]["text"] == "Hello world"
        print("✓ Test 1 passed: single speaker")
        passed += 1
    except Exception as e:
        print(f"✗ Test 1 FAILED: {e}")
        failed += 1
    
    # Test 2: Two speakers
    try:
        segments = [
            {"id": 0, "start": 0.0, "end": 4.0, "text": "Speaker A talking"},
            {"id": 1, "start": 5.0, "end": 9.0, "text": "Speaker B talking"},
            {"id": 2, "start": 10.0, "end": 14.0, "text": "Speaker A again"},
        ]
        diarization = [
            {"start_s": 0.0, "end_s": 4.5, "speaker": "SPEAKER_00"},
            {"start_s": 4.5, "end_s": 9.5, "speaker": "SPEAKER_01"},
            {"start_s": 9.5, "end_s": 15.0, "speaker": "SPEAKER_00"},
        ]
        result = assign_speakers(segments, diarization)
        assert result[0]["speaker"] == "SPEAKER_00"
        assert result[1]["speaker"] == "SPEAKER_01"
        assert result[2]["speaker"] == "SPEAKER_00"
        print("✓ Test 2 passed: two speakers")
        passed += 1
    except Exception as e:
        print(f"✗ Test 2 FAILED: {e}")
        failed += 1
    
    # Test 3: Empty diarization defaults to SPEAKER_00
    try:
        segments = [{"id": 0, "start": 0.0, "end": 3.0, "text": "Hello"}]
        result = assign_speakers(segments, [])
        assert result[0]["speaker"] == "SPEAKER_00"
        print("✓ Test 3 passed: empty diarization")
        passed += 1
    except Exception as e:
        print(f"✗ Test 3 FAILED: {e}")
        failed += 1
    
    # Test 4: Does not mutate input
    try:
        segments = [{"id": 0, "start": 0.0, "end": 3.0, "text": "Hello"}]
        original = segments[0].copy()
        assign_speakers(segments, [])
        assert segments[0] == original
        assert "speaker" not in segments[0]
        print("✓ Test 4 passed: no mutation")
        passed += 1
    except Exception as e:
        print(f"✗ Test 4 FAILED: {e}")
        failed += 1
    
    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run — expected to FAIL at this point
run_tests()

✓ Test 1 passed: single speaker
✓ Test 2 passed: two speakers
✓ Test 3 passed: empty diarization
✓ Test 4 passed: no mutation

Results: 4 passed, 0 failed


True

### 1.3 — Implement `assign_speakers`

Open `foreign_whispers/diarization.py` and add the function below at the bottom of the file. Replace the `raise NotImplementedError` with your implementation.

**Hints:**
1. Create a **copy** of each segment dict (don't mutate the input)
2. For each segment, compute overlap with every diarization interval:
   ```python
   overlap = max(0, min(seg_end, diar_end) - max(seg_start, diar_start))
   ```
3. Pick the diarization interval with the **largest overlap**
4. If no overlap or diarization is empty, default to `"SPEAKER_00"`

In [13]:
# Stub — copy this into foreign_whispers/diarization.py (at the bottom)

def assign_speakers(
    segments: list[dict],
    diarization: list[dict],
) -> list[dict]:
    """Assign a speaker label to each transcription segment.

    For each segment, finds the diarization interval with the greatest
    temporal overlap and copies its speaker label. If diarization is
    empty, all segments default to ``SPEAKER_00``.

    Args:
        segments: Whisper-style ``[{id, start, end, text, ...}]``.
        diarization: pyannote-style ``[{start_s, end_s, speaker}]``.

    Returns:
        New list of segment dicts, each with an added ``speaker`` key.
        Original list is not mutated.
    """
    # ---- YOUR CODE HERE ----
    raise NotImplementedError("Implement this function")
    # ---- END YOUR CODE ----

### 1.4 — Re-run the tests

After implementing, re-run the tests. All 4 should pass.

In [14]:
# Reload the module and re-run
import importlib
import foreign_whispers.diarization
importlib.reload(foreign_whispers.diarization)

assert run_tests(), "Some tests failed — fix your implementation before continuing."

✓ Test 1 passed: single speaker
✓ Test 2 passed: two speakers
✓ Test 3 passed: empty diarization
✓ Test 4 passed: no mutation

Results: 4 passed, 0 failed


### 1.5 — Commit your work

```bash
git add foreign_whispers/diarization.py
git commit -m "feat: add assign_speakers merge function"
```

---

## Task 2: Diarize API Endpoint

**Goal:** Create `POST /api/diarize/{video_id}` that extracts audio, runs diarization, and returns speaker segments.

**Files to create:**
- `api/src/schemas/diarize.py`
- `api/src/routers/diarize.py`

**Files to modify:**
- `api/src/main.py` (register the router)
- `api/src/core/config.py` (add `diarizations_dir` property)

### 2.1 — Add `diarizations_dir` to Settings

Open `api/src/core/config.py` and add this property after `transcriptions_dir`:

```python
@property
def diarizations_dir(self) -> Path:
    return self.data_dir / "diarizations"
```

### 2.2 — Create the response schema

Create `api/src/schemas/diarize.py`:

In [15]:
# Preview of the schema — create this file at api/src/schemas/diarize.py

schema_code = '''
"""Pydantic schemas for the diarize API contract."""

from pydantic import BaseModel


class DiarizeSpeakerSegment(BaseModel):
    start_s: float
    end_s: float
    speaker: str


class DiarizeResponse(BaseModel):
    video_id: str
    speakers: list[str]
    segments: list[DiarizeSpeakerSegment]
    skipped: bool = False
'''

schema_path = PROJECT_ROOT / "api" / "src" / "schemas" / "diarize.py"
schema_path.write_text(schema_code.strip() + "\n")
print(f"Created {schema_path}")

Created /home/joshu/CS4613/foreign-whispers/api/src/schemas/diarize.py


### 2.3 — Create the router (with stub)

Create `api/src/routers/diarize.py`. The endpoint has a stub that returns `501 Not Implemented` — you will fill it in.

In [ ]:
router_code = '''
"""POST /api/diarize/{video_id} — speaker diarization (issue fw-lua)."""

import asyncio
import json
import subprocess

from fastapi import APIRouter, HTTPException

from api.src.core.config import settings
from api.src.core.dependencies import resolve_title
from api.src.schemas.diarize import DiarizeResponse
from api.src.services.alignment_service import AlignmentService

router = APIRouter(prefix="/api")

_alignment_service = AlignmentService(settings=settings)


@router.post("/diarize/{video_id}", response_model=DiarizeResponse)
async def diarize_endpoint(video_id: str):
    """Run speaker diarization on a video's audio track.

    Steps:
    1. Extract audio from video via ffmpeg
    2. Run pyannote diarization
    3. Cache and return speaker segments
    """
    title = resolve_title(video_id)
    if title is None:
        raise HTTPException(status_code=404, detail=f"Video {video_id} not found")

    diar_dir = settings.diarizations_dir
    diar_dir.mkdir(parents=True, exist_ok=True)
    diar_path = diar_dir / f"{title}.json"

    # Return cached result
    if diar_path.exists():
        data = json.loads(diar_path.read_text())
        return DiarizeResponse(
            video_id=video_id,
            speakers=data.get("speakers", []),
            segments=data.get("segments", []),
            skipped=True,
        )

    # ---- YOUR CODE HERE ----
    video_path = settings.videos_dir / f"{title}.mp4"
    audio_path = diar_dir / f"{title}.wav"
    subprocess.run(
        ["ffmpeg", "-i", str(video_path), "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-y", str(audio_path)],
        check=True,
    )
    # Step 2: Run diarization
    diar_segments = _alignment_service.diarize(str(audio_path))
    #
    # Step 3: Extract unique speakers
    speakers = sorted(set(s["speaker"] for s in diar_segments))
    #
    # Step 4: Cache result
    result = {"speakers": speakers, "segments": diar_segments}
    diar_path.write_text(json.dumps(result))
    #

    from foreign_whispers.diarization import assign_speakers

    transcript_path = settings.transcriptions_dir / f"{title}.json"
    if transcript_path.exists():
        transcript = json.loads(transcript_path.read_text())
        labeled_segments = assign_speakers(transcript.get("segments", []), diar_segments)
        transcript["segments"] = labeled_segments
        transcript_path.write_text(json.dumps(transcript))
        
    # Step 5: Return DiarizeResponse
    return DiarizeResponse(video_id=video_id, speakers=speakers, segments=diar_segments)
'''

router_path = PROJECT_ROOT / "api" / "src" / "routers" / "diarize.py"
router_path.write_text(router_code.strip() + "\n")
print(f"Created {router_path}")

Created /home/joshu/CS4613/foreign-whispers/api/src/routers/diarize.py


### 2.4 — Register the router

Open `api/src/main.py` and add these lines alongside the existing router registrations (around line 94):

```python
from api.src.routers.diarize import router as diarize_router
app.include_router(diarize_router)
```

### 2.5 — Implement the endpoint

Replace the `YOUR CODE HERE` block in `api/src/routers/diarize.py` with your implementation. Follow the numbered steps in the comments.

### 2.6 — Rebuild and test

In [17]:
# Rebuild the API container
!cd {PROJECT_ROOT} && docker compose --profile nvidia build api
!cd {PROJECT_ROOT} && docker compose --profile nvidia up -d api


[+] Building 0.0s (0/1)                                                         
 => [internal] load local bake definitions                                 0.0s
[+] Building 0.2s (1/1)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 608B                                             0.0s
[+] Building 0.2s (1/2)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 608B                                             0.0s
[+] Building 0.5s (2/4)                                                         
 => [internal] load local bake definitions                                 0.0s
 => => reading from stdin 608B                                             0.0s
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 1.1

In [18]:
import requests

API_BASE = "http://localhost:8080"

# Get the first video ID from the registry
videos = requests.get(f"{API_BASE}/api/videos").json()
video_id = videos[0]["id"]
print(f"Testing with video: {videos[0]['title']} ({video_id})")

# Call the diarize endpoint
resp = requests.post(f"{API_BASE}/api/diarize/{video_id}")
print(f"Status: {resp.status_code}")
print(resp.json())

ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

### 2.7 — Commit

```bash
git add api/src/schemas/diarize.py api/src/routers/diarize.py api/src/main.py api/src/core/config.py
git commit -m "feat: add POST /api/diarize endpoint with caching"
```

---

## Task 3: Merge Speaker Labels Into Transcription

**Goal:** After diarization runs, update the transcription JSON so each segment has a `speaker` field. This way downstream stages (translate, TTS) know which speaker said what.

**File to modify:** `api/src/routers/diarize.py`

### 3.1 — Add merge step to the diarize endpoint

After your diarization result is cached, add this code to update the transcription JSON with speaker labels:

```python
from foreign_whispers.diarization import assign_speakers

transcript_path = settings.transcriptions_dir / f"{title}.json"
if transcript_path.exists():
    transcript = json.loads(transcript_path.read_text())
    labeled_segments = assign_speakers(transcript.get("segments", []), diar_segments)
    transcript["segments"] = labeled_segments
    transcript_path.write_text(json.dumps(transcript))
```

### 3.2 — Verify the merge

In [ ]:
import json

# After running diarize, check the transcription for speaker labels
# Replace TITLE with your video's title
transcription_dir = PROJECT_ROOT / "pipeline_data" / "api" / "transcriptions" / "whisper"
transcription_files = list(transcription_dir.glob("*.json"))

if transcription_files:
    data = json.loads(transcription_files[0].read_text())
    print(f"File: {transcription_files[0].name}")
    print(f"Number of segments: {len(data.get('segments', []))}")
    print("\nFirst 3 segments:")
    for seg in data.get("segments", [])[:3]:
        speaker = seg.get("speaker", "<NO SPEAKER>")
        print(f"  [{seg['start']:.1f}s - {seg['end']:.1f}s] {speaker}: {seg['text'][:60]}")
else:
    print("No transcription files found — run the pipeline first.")

### 3.3 — Commit

```bash
git add api/src/routers/diarize.py
git commit -m "feat: merge speaker labels into transcription after diarization"
```

---

## Task 4: Frontend Pipeline Integration

**Goal:** Call the diarize endpoint from the frontend pipeline when diarization is enabled in settings.

**Files to modify:**
- `frontend/src/lib/api.ts` — add `diarizeVideo` function
- `frontend/src/lib/types.ts` — add `"diarize"` to `PipelineStage`
- `frontend/src/hooks/use-pipeline.ts` — call diarize between transcribe and translate
- `frontend/src/components/pipeline-table.tsx` — add diarize row
- `frontend/src/components/pipeline-status-bar.tsx` — add status message

### 4.1 — Add the API client function

In `frontend/src/lib/api.ts`, add:

```typescript
export async function diarizeVideo(videoId: string): Promise<DiarizeResponse> {
  return fetchJson<DiarizeResponse>(`/api/diarize/${videoId}`, {
    method: "POST",
  });
}
```

### 4.2 — Add TypeScript types

In `frontend/src/lib/types.ts`, add:

```typescript
export interface DiarizeResponse {
  video_id: string;
  speakers: string[];
  segments: { start_s: number; end_s: number; speaker: string }[];
  skipped: boolean;
}
```

Update `PipelineStage`:

```typescript
export type PipelineStage = "download" | "transcribe" | "diarize" | "translate" | "tts" | "stitch";
```

### 4.3 — Wire into the pipeline hook

In `frontend/src/hooks/use-pipeline.ts`, add the diarize call **between** transcribe and translate:

```typescript
// After: await run("transcribe", () => transcribeVideo(dl.video_id, settings.useYoutubeCaptions));
// Before: await run("translate", () => translateVideo(dl.video_id, "es"));

if (settings.diarization.length > 0) {
  await run("diarize", () => diarizeVideo(dl.video_id));
}
```

Don't forget to import `diarizeVideo` at the top of the file.

### 4.4 — Add UI elements

In `frontend/src/components/pipeline-table.tsx`, add a row to `STAGES`:

```typescript
import { UsersIcon } from "lucide-react";

// Add after "transcribe" entry:
{ key: "diarize", label: "Diarize", icon: UsersIcon, description: "Speaker diarization via pyannote" },
```

In `frontend/src/components/pipeline-status-bar.tsx`, add:

```typescript
diarize: "Running pyannote speaker diarization...",
```

### 4.5 — Build and test

In [ ]:
# Rebuild the frontend
!cd {PROJECT_ROOT} && docker compose --profile nvidia build frontend
!cd {PROJECT_ROOT} && docker compose --profile nvidia up -d frontend

print("\nOpen http://localhost:8501")
print("1. Click the gear icon to open Settings")
print("2. Go to TTS > Diarization and select 'pyannote'")
print("3. Run the pipeline — the Diarize stage should appear in the table")

### 4.6 — Commit

```bash
git add frontend/src/lib/api.ts frontend/src/lib/types.ts \
  frontend/src/hooks/use-pipeline.ts \
  frontend/src/components/pipeline-table.tsx \
  frontend/src/components/pipeline-status-bar.tsx
git commit -m "feat: add diarize stage to frontend pipeline"
```

---

## Task 5: Per-Speaker TTS Voice Selection

**Goal:** When speaker labels exist in the translated segments, use a different Chatterbox reference voice per speaker.

**Files to modify:**
- `api/src/routers/tts.py`
- `api/src/services/tts_service.py`

**Approach:**
1. Read the translated JSON — each segment now has a `speaker` field
2. Map each unique speaker to a reference WAV from `pipeline_data/speakers/{lang}/`
3. Pass the speaker-to-voice mapping to the TTS engine so it switches voices per segment

In [ ]:
# Explore available speaker reference voices
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
print("Available languages:")
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = list(lang_dir.glob("*.wav"))
        print(f"  {lang_dir.name}/  ({len(wavs)} reference WAVs)")
        for wav in wavs[:3]:
            print(f"    - {wav.name}")

### 5.1 — Design your voice assignment strategy

Some options to consider:
- **Round-robin:** Assign voices in order (SPEAKER_00 → voice_1.wav, SPEAKER_01 → voice_2.wav)
- **Filename-based:** Match speaker index to file index
- **Default + override:** Use `default.wav` for all speakers unless specific speaker WAVs are provided

Document your chosen approach in the cell below:

*YOUR DESIGN NOTES HERE*

- Strategy chosen: ...
- How speakers map to voice files: ...
- Edge cases considered: ...

### 5.2 — Implement and test

Modify `tts_service.py` to accept a speaker-to-voice mapping and switch reference audio per segment. This is open-ended — use the existing `text_file_to_speech` function as your starting point.

---

## Evaluation Criteria

| # | Criterion | How to verify |
| - | --------- | ------------- |
| 1 | Tests pass | Re-run the test cell in Task 1.4 — all 4 green |
| 2 | API works | `POST /api/diarize/{video_id}` returns speaker segments |
| 3 | Merge works | Transcription JSON has `speaker` fields after diarization |
| 4 | Frontend works | Diarize stage appears in pipeline table when enabled |
| 5 | Caching works | Second API call returns `skipped: true` |
| 6 | Code quality | Follows existing patterns (file-exists cache, service layer, Pydantic schemas) |